# Terrain Flux: build preset terrain

Turns the GeoTIFFs from `export_presets_gee.js` into the compact presets baked into `index.html`.

**Steps:** run the cells top to bottom. Upload your `tf_*.tif` files when asked, then copy the printed lines into the `PRESETS` list in `index.html`.

Each preset is the center square of the DEM (minus a 3% edge trim), box-averaged to 129 by 129 corners, stored as 16-bit meters and base64 encoded. About 44 KB per place.

In [ ]:
!pip install -q rasterio

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick your tf_*.tif files

## Settings

`ocean`: True if the place touches the sea. For inland places any zero values (Earth Engine fills the corners of a reprojected box with 0) get replaced with the median so they do not turn into ponds.

`relief`: how many height steps the place gets in the game. Taller, steeper places want more.

In [ ]:
PLACES = {
    # file stem:        (display name,      icon, ocean, relief, blurb)
    'morro_bay':        ('Morro Bay',       '🪨', True,  18, 'The Rock, the sandspit, and the estuary.'),
    'big_sur':          ('Big Sur',         '🌊', True,  26, 'Mountains that skipped the foothills and went straight into the Pacific.'),
    'yosemite_valley':  ('Yosemite Valley', '🧗', False, 26, 'El Capitan, Half Dome, and a very flat valley floor for your suburbs.'),
    'san_francisco':    ('San Francisco',   '🌉', True,  16, 'The Golden Gate, the hills, and zero room to build. Good luck.'),
}

In [ ]:
import base64
import numpy as np
import rasterio
from PIL import Image

NC = 129  # corners per side (128 tiles)

def build(stem, name, icon, ocean, relief, blurb):
    with rasterio.open(f'tf_{stem}.tif') as src:
        a = src.read(1).astype(np.float32)
        px = abs(src.transform.a)  # pixel size in meters
    h, w = a.shape
    s = min(h, w)
    inset = int(s * 0.03)
    s2 = s - 2 * inset
    y0, x0 = (h - s) // 2 + inset, (w - s) // 2 + inset
    a = a[y0:y0 + s2, x0:x0 + s2]
    if not ocean:
        bad = a <= 0
        if bad.any():
            a[bad] = np.median(a[~bad])
    r = np.array(Image.fromarray(a, mode='F').resize((NC, NC), Image.BOX))
    r = np.round(r).astype('<i2')
    km = round(s2 * px / 1000, 1)
    b64 = base64.b64encode(r.tobytes()).decode()
    print(f'# {name}: {r.min()} to {r.max()} m, {km} km across, {(r <= 0).mean():.0%} sea')
    return (f"  {{ id: '{stem}', name: '{name}', icon: '{icon}', blurb: '{blurb}', "
            f"relief: {relief}, km: {km}, data: '{b64}' }}")

lines = [build(stem, *cfg) for stem, cfg in PLACES.items()]
js = 'const PRESETS = [\n' + ',\n'.join(lines) + '\n];\n'
open('presets.js', 'w').write(js)
print(f'\nWrote presets.js ({len(js) / 1024:.0f} KB). Paste it over the PRESETS list in index.html.')

In [ ]:
files.download('presets.js')